# 3DEP Extras — AOI Comparison & Shift

Compare terrain feature distributions across AOIs before modeling.

In [ ]:
import ee, numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from scipy.stats import wasserstein_distance
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

ee.Authenticate()
ee.Initialize(project="shrubwise-dc-488219")
THREEDEP_10M = ee.ImageCollection("USGS/3DEP/10m_collection")
AOI_BLISS = ee.Geometry.Rectangle([-120.1018846, 38.99274873, -120.0899834, 39.0020357], geodesic=False)


In [ ]:
def dem_feature_stack():
    dem = THREEDEP_10M.mosaic().select("elevation").toFloat().rename("elevation")
    slope = ee.Terrain.slope(dem).rename("slope_deg")
    aspect = ee.Terrain.aspect(dem).rename("aspect_deg")
    return ee.Image.cat([dem, slope, aspect])

def compare_band(image: ee.Image, band: str, aois: dict, *, n_samples=5000, scale=10, seed=42):
    img = image.select(band).toFloat()
    def sample_vals(geom):
        pts = ee.FeatureCollection.randomPoints(region=geom, points=n_samples, seed=seed)
        vals = img.sampleRegions(pts, scale=scale, geometries=False).aggregate_array(band).getInfo()
        arr = np.array(vals, dtype=float)
        return arr[np.isfinite(arr)]
    samples = {name: sample_vals(geom) for name, geom in aois.items()}
    rows = []
    for name, arr in samples.items():
        rows.append({"aoi": name, "n": arr.size, "mean": float(arr.mean()) if arr.size else np.nan,
                     "median": float(np.median(arr)) if arr.size else np.nan})
    summary = pd.DataFrame(rows).set_index("aoi")
    names = list(samples)
    shift = None
    if len(names) >= 2:
        a, b = samples[names[0]], samples[names[1]]
        if a.size and b.size:
            if _HAS_SCIPY:
                w = float(wasserstein_distance(a, b))
            else:
                qs = np.linspace(0,1,101)
                w = float(np.mean(np.abs(np.quantile(a,qs)-np.quantile(b,qs))))
            shift = {"aoi_a": names[0], "aoi_b": names[1], "wasserstein": w}
    return summary, shift, samples

# Demo second AOI shifted east
minx, miny, maxx, maxy = -120.1018846, 38.99274873, -120.0899834, 39.0020357
center_lat = (miny+maxy)/2
km_shift = 10
dlon = km_shift / (111.32 * np.cos(np.deg2rad(center_lat)))
AOI_SHIFT = ee.Geometry.Rectangle([minx+dlon, miny, maxx+dlon, maxy], geodesic=False)
AOIS = {"DL_Bliss": AOI_BLISS, "Demo_Shifted_10km": AOI_SHIFT}
img = dem_feature_stack()
summary, shift, samples = compare_band(img, "elevation", AOIS)
print(summary)
print(shift)
plt.figure()
for name, arr in samples.items():
    plt.hist(arr, bins=40, alpha=0.5, label=name)
plt.title("3DEP elevation comparison")
plt.xlabel("Elevation (m)")
plt.ylabel("Count")
plt.legend()
plt.show()
